# ANLP Project — Kaggle Session 1

**Scope:** BART fine-tuning + inference + all 6 prompting conditions + evaluation.

**Estimated time:** ~6 hours on T4.

**Before running:**
- Sidebar → Accelerator → **GPU T4 x2**
- Sidebar → Internet → **On**

**At the end:** click **Save Version → Save & Run All** so `/kaggle/working` is snapshotted. Then attach this snapshot as a *Dataset* input to Session 2 so the combined evaluation table includes BART results.

## 1. Clone repo (dataset is bundled)

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --quiet --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!ls

## 2. Install dependencies

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

## 3. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi -L

## 4. Fine-tune BART (~2–4 h)

Two-phase training (encoder frozen → unfrozen), early stopping on val ROUGE-L. Checkpoint saved to `checkpoints/bart/`.

In [ ]:
!python scripts/run_finetuning.py --model bart --output_dir checkpoints/bart

## 5. BART inference on 9 test matches (~10 min)

In [ ]:
!python scripts/run_inference_finetuned.py --model bart --checkpoint checkpoints/bart

## 6. All 6 prompting conditions (~3–5 h total)

Each condition is independent and saves its own JSON. If you run out of time, comment out remaining conditions — partial results still evaluate.

In [ ]:
!python scripts/run_prompting.py --model flan --strategy chunk --prompt zero
!python scripts/run_prompting.py --model flan --strategy chunk --prompt few
!python scripts/run_prompting.py --model flan --strategy chunk --prompt cot
!python scripts/run_prompting.py --model led  --strategy long  --prompt zero
!python scripts/run_prompting.py --model led  --strategy long  --prompt few
!python scripts/run_prompting.py --model led  --strategy long  --prompt cot

## 7. Evaluate everything done so far

In [ ]:
# BERTScore is buggy on the current Kaggle transformers/bert_score combo (OverflowError).
# Use --no-bertscore for ROUGE-only metrics.
!python scripts/run_evaluation.py --no-bertscore

## 8. Inspect results

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:600]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)
    if p.endswith('.csv'):
        print(open(p).read())

## 9. Save for Session 2

Now click **Save Version → Save & Run All** in the top right.

After it finishes (note: "Save Version" re-runs the whole notebook from scratch — for that reason it can be smarter to do **Save Version → Quick Save** instead, which just snapshots the current state without re-running).

Then in Session 2 notebook: sidebar → **Add Data → Your Datasets** → search for the output of this kernel and attach it. Session 2 will read `outputs/predictions/*.json` and `checkpoints/bart/` from there if needed.